Changes from previous crawler system:
- crawler: extract_text now also focuses only on the html element "entry.clearfix"
- chunking changed so that it has overlap among chunks and so that each chunk fits the context window of the small embedding model (256 tokens ~ 150-200 words ~ 800 characters)
- embedding model changed (previous one was not good with languages other than english)
- pdf reader library changed from pypdf2 to pdfplumber (better at reading tables)

In [ ]:
import requests   # e "browserul" codului; folosim pentru a trimite cereri HTTP si a primi raspunsul
from bs4 import BeautifulSoup  # instrument de parsing; BS transforma html brut intr un format lizibil pt Python
from urllib.parse import urljoin, urlparse
#urljoin: lipeste fragmentele de link. De ex. daca esti pe google.com si gasesti un link catre /images,
#         il transforma automat in linkul complet: https://google.com/images
#urlparse: exact opusul; descompune un URL in bucatele (protocol, domeniu, cale), ceea ce ajuta la
#          a identifica exact de unde vine un link
import io
import pdfplumber   # pentru a citi si din pdf-uri, nu doar text html


visited = set()
MAX_DEPTH = 2
ALLOWED_PREFIXES = [
    "/admitere/",
    "/regulamentul-concursului"
]

# Verifica daca linkul "link" gasit pe o alta pagina apartine aceluiasi site de baza, cel de Admitere UBB,
# sau daca este un link extern care te scoate de pe site.
# Literalmente: descompune ambele linkuri si le compara componenta de domeniu (admitere.ubbcluj.ro)
def is_internal(link, base):
    return urlparse(link).netloc == urlparse(base).netloc

def is_relevant_link(link):
    parsed = urlparse(link)

    path = parsed.path.lower()

    return any(prefix in path for prefix in ALLOWED_PREFIXES)


def extract_text(html):
    soup = BeautifulSoup(html, "html.parser") #transforma textul brut intr-un obiect structurat (un "copac" de elemente)

    main_content = soup.find("div", class_="entry clearfix")

    if not main_content:
        return ""

    # Nu avem nevoie de textul din interiorul tagurilor urmatoare, deci le stergem cu totul
    for tag in main_content(["script", "style", "nav", "footer"]): 
        tag.decompose()

    # get_text ne da "Hello!" in loc de "<p>Hello!</p>"
    return main_content.get_text(separator=" ", strip=True)

def extract_text_from_pdf(content):
    with pdfplumber.open(io.BytesIO(content)) as pdf:
        text = ""
        for page in pdf.pages:
            # Extrage tabelele și le formatează frumos ca text
            table = page.extract_table()
            if table:
                for row in table:
                    text += " | ".join([str(cell) if cell else "" for cell in row]) + "\n"
            # Extrage și restul textului
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text


def extract_links(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    links = []

    # Gaseste sectiunea principala a paginii pt a evita linkuri irelevante
    main_content = soup.find("div", class_="entry clearfix")

    if not main_content:
        return []

    for a in main_content.find_all("a", href=True): # cauta toate linkurile (eticheta <a>)
        link = urljoin(base_url, a["href"]) # pentru cazul ca e link relativ; daca in pagina scrie doar <a href="contact.html">, il transforma in https://admitere.ubbcluj.ro/contact.html
        
        if link.lower().endswith(".pdf"):  # evitam pdf-urile cu subiecte sau bareme de concurs
            if any(lang in link.lower() for lang in ["subiecte"]):
                continue
        if is_internal(link, base_url) and is_relevant_link(link):
            links.append(link)

    return links


def crawl(url, depth=0):
    # sa nu accesam prea multe sublinkuri & sa nu le repetam pe aceleasi
    if depth > MAX_DEPTH or url in visited:
        return []

    visited.add(url)

    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/119.0.0.0"} # ca sa nu ne blocheze serverul crezand ca suntem un bot
        response = requests.get(url, headers=headers, timeout=10, stream=True)  # stream=True: descarca intai informatiile despre fisier si decide dupa daca l si descarca pt a i extrage textul

        # --VERIFICARE: Vrem ori HTML, ori pdf
        content_type = response.headers.get('Content-Type', '').lower()

        # Cazul 1: Este pagina HTML
        if 'text/html' in content_type:
            print('Vizitez url: ',url)
            html = response.text        # textul insusi, cu taguri cu tot
            text = extract_text(html)   # textul fara taguri si filtrat
            
            results = []
            if text and len(text.strip()) > 100:
                results.append(text)
            for link in extract_links(html, url):
                results += crawl(link, depth + 1)

            return results
        
        # # Cazul 2: Este pdf
        elif 'application/pdf' in content_type:
            print(f"Extrag din pdf: ", url)
            text = extract_text_from_pdf(response.content)  # folosim .content (bytes), nu text
            return [text]

        # Cazul 3: Orice altceva, ignoram
        else:
            return []
        # --
        
    except Exception as e:
        print(f"Eroare la {url}: {e}")
        return []

In [7]:
urls = [
    "https://www.cs.ubbcluj.ro/admitere/nivel-licenta/admiterea-la-facultatea-de-matematica-si-informatica-nivel-licenta/",
    "https://www.cs.ubbcluj.ro/regulamentul-concursului-mate-info-ubb-2026/",
    "https://www.cs.ubbcluj.ro/admitere/nivel-licenta/calendarul-concursului-de-admitere-nivel-licenta/",
    "https://www.cs.ubbcluj.ro/admitere/nivel-licenta/acte-necesare-la-inscrierea-la-facultate-nivel-licenta/",
    "https://www.cs.ubbcluj.ro/admitere/nivel-licenta/taxe-admitere-facultate-nivel-licenta/",
]

docs = []

for url in urls:
    docs.extend(crawl(url))

docs = list(set(docs))

Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/admiterea-la-facultatea-de-matematica-si-informatica-nivel-licenta/
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/tematica-admitere-facultate/
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/taxe-admitere-facultate-nivel-licenta/
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/calendarul-concursului-de-admitere-nivel-licenta/
Vizitez url:  https://www.cs.ubbcluj.ro/regulamentul-concursului-mate-info-ubb-2026/
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/acte-necesare-la-inscrierea-la-facultate-nivel-licenta/


In [ ]:
def semantic_chunk_text(text, chunk_size=800, chunk_overlap=150):
    if len(text) <= chunk_size:
        return [text.strip()]

    separators = ["\n\n", "\n", ". ", " "]
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        if end >= len(text):
            chunks.append(text[start:].strip())
            break
            
        chosen_end = end
        for sep in separators:
            last_sep_pos = text.rfind(sep, start, end)
            if last_sep_pos != -1:
                chosen_end = last_sep_pos + len(sep)
                break
        
        chunk = text[start:chosen_end].strip()
        if chunk:
            chunks.append(chunk)
            
        # Calculăm overlap-ul
        theoretical_next_start = chosen_end - chunk_overlap
        actual_next_start = -1
        
        for sep in separators:
            first_sep_pos = text.find(sep, max(0, theoretical_next_start), chosen_end)
            if first_sep_pos != -1:
                actual_next_start = first_sep_pos + len(sep)
                break
        
        # Dacă nu găsim separator sau noul start nu avansează, forțăm avansul
        if actual_next_start <= start:
            start = chosen_end 
        else:
            start = actual_next_start
            
    return [c for c in chunks if len(c) > 5]

In [1]:
import re

def clean_text(text):
    # Înlocuiește 3 sau mai multe newline-uri cu exact două (păstrează paragraful)
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Înlocuiește spațiile multiple (orizontale) cu un singur spațiu
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

chunks = []
for doc in docs:
    # 1. Curățăm textul (foarte important pentru PDF-uri)
    cleaned_doc = clean_text(doc)
    
    # 2. Aplicăm chunking-ul cu overlap
    # Folosim chunk_size=800 pentru a nu depăși fereastra modelului de embedding
    doc_chunks = semantic_chunk_text(cleaned_doc)
    
    chunks.extend(doc_chunks)

print(f"Total chunks în baza de date: {len(chunks)}")

NameError: name 'docs' is not defined

In [20]:
# EMBEDDING

from sentence_transformers import SentenceTransformer
import faiss  # baza de date specializata in viteza. Daca ai 1 milion de fragmente de text, FAISS (Facebook AI Similarity Search) gaseste instant care dintre ele seamana cu intrebarea ta
import numpy as np  # FAISS lucreaza doar cu structuri de date de tip numpy

#if model not in globals():   # daca nu am definit pana acum model
model = SentenceTransformer("intfloat/multilingual-e5-small")

# Adaugă prefixul obligatoriu pentru modelul E5 la indexare
chunks_with_prefix = [f"passage: {c}" for c in chunks]
embeddings = model.encode(chunks)

# Creeaza un index (memorie rapida) care masoara distanta "L2" (distanta euclidiana, ca o linie dreapta pe harta) intre vectori
index = faiss.IndexFlatL2(len(embeddings[0]))  # len(embeddings[0]) -> ii spune indexului cat de lung este un singur vector (pentru acest model in specific lungimea fiind de 384 de numere)
index.add(np.array(embeddings))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11583.90it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
# SAVE
np.save("Data/chunks.npy", chunks)
faiss.write_index(index, "Data/index.faiss")